# Willett Reconstruction Baseline

Minimal Colab notebook for training the native-cache Willett-style GRU baseline and plotting convergence diagnostics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

REPO_DIR = Path('/content/utah-ssl')
REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only || true

%cd {REPO_DIR}
!pip install -q torch pandas matplotlib

RAW_CACHE_ROOT = Path('/content/drive/MyDrive/utah_ssl/data/cache_v1')
SMOOTHED_CACHE_ROOT = Path('/content/drive/MyDrive/utah_ssl/data/cache_v1_smoothed_sigma2p0')
OUTPUT_ROOT = Path('/content/drive/MyDrive/utah_ssl/outputs/willett_reconstruction')
PYTHONPATH = str(REPO_DIR / 'analysis' / 'active' / 'ssl_experiments')
os.environ['PYTHONPATH'] = PYTHONPATH

In [ ]:
from pathlib import Path

RUN_NAME = 'willett_tx_only_colab'
DATASET = 'brain2text24'
FEATURE_MODE = 'tx_only'
CACHE_ROOT = RAW_CACHE_ROOT
MAX_STEPS = 5000
BATCH_SIZE = 64
SESSION_ADAPTER = True
RESUME_LATEST = True
PRECOMPUTED_STATS_PATH = None

RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR

In [ ]:
import os
import shlex

cmd = [
    'python', '-m', 'willett_reconstruction.train',
    '--cache-root', str(CACHE_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--run-name', RUN_NAME,
    '--dataset', DATASET,
    '--feature-mode', FEATURE_MODE,
    '--max-steps', str(MAX_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--val-every-steps', '100',
    '--checkpoint-every-steps', '500',
    '--progress-every-steps', '25',
]
if RESUME_LATEST:
    cmd.append('--resume-latest')
if not SESSION_ADAPTER:
    cmd.append('--disable-session-adapter')
if PRECOMPUTED_STATS_PATH is not None:
    cmd.extend(['--precomputed-split-stats-path', str(PRECOMPUTED_STATS_PATH)])

full_cmd = ' '.join(shlex.quote(part) for part in cmd)
print(full_cmd)
!PYTHONPATH={PYTHONPATH} {full_cmd}

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

summary_path = RUN_DIR / 'summary.json'
progress_path = RUN_DIR / 'progress.jsonl'
summary = json.loads(summary_path.read_text())
progress = [json.loads(line) for line in progress_path.read_text().splitlines() if line.strip()]
progress_df = pd.DataFrame(progress)
train_df = progress_df[progress_df['event'] == 'willett_train_report'].copy()
val_df = progress_df[progress_df['event'] == 'willett_val_report'].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
if not train_df.empty:
    axes[0].plot(train_df['step'], train_df['train_ctc_bpphone'], label='train CTC')
if not val_df.empty:
    axes[0].plot(val_df['step'], val_df['val_ctc_bpphone'], marker='o', label='val CTC')
    axes[1].plot(val_df['step'], val_df['val_phoneme_error_rate'], marker='o', label='val PER')
axes[0].set_title('Willett Reconstruction CTC')
axes[0].set_xlabel('step')
axes[0].set_ylabel('bits / phoneme')
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].set_title('Willett Reconstruction PER')
axes[1].set_xlabel('step')
axes[1].set_ylabel('PER')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

print('best step:', summary.get('best_step'))
print('final metrics:', json.dumps(summary.get('metrics', {}), indent=2))
print('best metrics:', json.dumps(summary.get('best_metrics', {}), indent=2))
if not val_df.empty:
    display(val_df[['step', 'val_ctc_bpphone', 'val_phoneme_error_rate']].tail(10))